# Module 8.2: AI智能体系统与模型上下文协议

## 概述

AI智能体（Agent）是能够自主感知环境、制定计划并执行动作以实现目标的系统。与传统的单次推理不同，智能体具有以下特点：

- **自主性**：能够独立决策和行动
- **目标导向**：围绕特定目标进行规划和执行
- **工具使用**：能够调用外部工具和API
- **迭代改进**：通过观察结果调整策略

**MCP (Model Context Protocol, 模型上下文协议)** 是一种标准化协议，用于规范大语言模型与外部工具、资源和数据源之间的通信接口。在本章场景中，它用于为智能体提供统一的工具调用和上下文管理能力。

## 学习目标

1. 理解AI智能体的核心架构和组件
2. 掌握ReAct模式（推理+行动）的实现
3. 学习模型上下文协议（MCP）的设计和应用
4. 实现智能体的规划和记忆系统
5. 构建多智能体协作系统
6. 了解智能体评估和安全机制
7. 应用智能体解决实际问题

### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- "查一下我的订单并改个地址"如何拆分步骤？→ Agent 的任务规划与工具调用
- 工具调用越权怎么控制？→ 权限边界与安全沙箱机制

## 智能体架构模式

```
┌─────────────────────────────────────┐
│         AI Agent System             │
├─────────────────────────────────────┤
│  Perception  →  Planning  →  Action │
│       ↑                        ↓    │
│       └────────  Memory  ──────┘    │
└─────────────────────────────────────┘
```

### 核心组件

1. **感知（Perception）**：理解环境和任务
2. **规划（Planning）**：制定行动计划
3. **行动（Action）**：执行工具调用
4. **记忆（Memory）**：维护上下文和经验

In [ ]:
# 导入必要的库
import numpy as np
import torch
import json
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum

np.random.seed(42)
torch.manual_seed(42)
print("✓ 库导入成功")

## 智能体理论基础

### 智能体循环

智能体的核心是一个持续的感知-思考-行动循环：

```
观察（Observe）→ 思考（Think）→ 行动（Act）→ 观察（Observe）→ ...
```

### 智能体类型

1. **简单反射型**：基于规则的直接映射
2. **基于模型**：维护环境模型进行推理
3. **目标导向**：根据目标评估行动
4. **效用导向**：最大化期望效用
5. **学习型**：从经验中改进策略

### 智能体能力

- **推理**：逻辑思考和问题分解
- **规划**：制定多步骤计划
- **工具使用**：调用外部API和函数
- **记忆管理**：存储和检索相关信息
- **自我反思**：评估和改进自身行为

In [ ]:
# 🔬 Micro Practice: Simple Agent Implementation
# Goal: 简单智能体实现

class SimpleAgent:
    """简单的任务执行智能体"""
    
    def __init__(self, name: str):
        self.name = name
        self.memory = []
        self.step_count = 0
    
    def perceive(self, task: str) -> Dict[str, Any]:
        """感知任务"""
        self.memory.append({"type": "task", "content": task})
        return {"task": task, "understood": True}
    
    def plan(self, task: str) -> List[str]:
        """制定计划"""
        # 简单的任务分解
        if "计算" in task or "calculate" in task.lower():
            steps = ["理解问题", "提取数字", "执行计算", "返回结果"]
        elif "搜索" in task or "search" in task.lower():
            steps = ["理解查询", "执行搜索", "筛选结果", "返回答案"]
        else:
            steps = ["分析任务", "执行操作", "验证结果"]
        
        self.memory.append({"type": "plan", "steps": steps})
        return steps
    
    def act(self, action: str) -> str:
        """执行动作"""
        self.step_count += 1
        result = f"执行步骤 {self.step_count}: {action}"
        self.memory.append({"type": "action", "content": action, "result": result})
        return result
    
    def execute(self, task: str) -> Dict[str, Any]:
        """完整的任务执行流程"""
        # 感知
        perception = self.perceive(task)
        print(f"[{self.name}] 感知: {perception['task']}")
        
        # 规划
        plan = self.plan(task)
        print(f"[{self.name}] 计划: {' → '.join(plan)}")
        
        # 执行
        results = []
        for step in plan:
            result = self.act(step)
            results.append(result)
            print(f"[{self.name}] {result}")
        
        return {
            "task": task,
            "plan": plan,
            "results": results,
            "steps": self.step_count
        }

# 测试简单智能体
agent = SimpleAgent("助手-01")
result = agent.execute("计算 123 + 456 的结果")

print(f"\n任务完成，共执行 {result['steps']} 个步骤")
print(f"记忆条目数: {len(agent.memory)}")

## ReAct模式：推理与行动

### ReAct原理

ReAct（Reasoning + Acting）是一种将推理和行动交织在一起的智能体模式。与传统的先思考后行动不同，ReAct在每一步都进行推理，根据观察结果动态调整策略。

### ReAct循环

```
Thought: 我需要找到X的信息
Action: search("X")
Observation: [搜索结果显示...]
Thought: 基于搜索结果，我应该计算Y
Action: calculate(Y)
Observation: [计算结果为...]
Thought: 现在我有足够信息回答问题
Action: finish(answer)
```

### ReAct优势

1. **可解释性**：每步推理过程清晰可见
2. **灵活性**：根据观察动态调整策略
3. **错误恢复**：可以从失败的行动中学习
4. **工具组合**：能够灵活组合多个工具

### 关键组件

- **Thought**：推理步骤，分析当前状态
- **Action**：执行的工具调用
- **Observation**：工具返回的结果
- **Parser**：解析LLM输出的思考和行动

In [ ]:
# 🔬 Micro Practice: ReAct Agent Implementation
# Goal: ReAct智能体实现

class ReActAgent:
    """实现ReAct模式的智能体"""
    
    def __init__(self, tools: Dict[str, callable], max_steps: int = 10):
        self.tools = tools
        self.max_steps = max_steps
        self.trace = []  # 记录思考-行动-观察轨迹
    
    def parse_action(self, text: str) -> tuple:
        """解析LLM输出的行动"""
        # 简化的解析逻辑
        if "Action:" in text:
            action_line = [line for line in text.split('\n') if 'Action:' in line][0]
            action_str = action_line.split('Action:')[1].strip()
            
            # 解析函数调用 tool_name(args)
            if '(' in action_str and ')' in action_str:
                tool_name = action_str.split('(')[0].strip()
                args_str = action_str.split('(')[1].split(')')[0]
                return tool_name, args_str
        return None, None
    
    def execute_action(self, tool_name: str, args: str) -> str:
        """执行工具调用"""
        if tool_name in self.tools:
            try:
                result = self.tools[tool_name](args)
                return str(result)
            except Exception as e:
                return f"Error: {str(e)}"
        return f"Tool '{tool_name}' not found"
    
    def run(self, task: str, simulate_llm: callable) -> Dict[str, Any]:
        """运行ReAct循环"""
        context = f"Task: {task}\n\n"
        
        for step in range(self.max_steps):
            # 模拟LLM生成思考和行动
            response = simulate_llm(context)
            
            # 记录思考
            thought = response.split('Action:')[0].replace('Thought:', '').strip()
            self.trace.append({"type": "thought", "content": thought})
            
            # 解析行动
            tool_name, args = self.parse_action(response)
            
            if tool_name == "finish":
                self.trace.append({"type": "finish", "answer": args})
                return {"success": True, "answer": args, "steps": step + 1, "trace": self.trace}
            
            if tool_name:
                self.trace.append({"type": "action", "tool": tool_name, "args": args})
                
                # 执行工具
                observation = self.execute_action(tool_name, args)
                self.trace.append({"type": "observation", "content": observation})
                
                # 更新上下文
                context += f"Thought: {thought}\n"
                context += f"Action: {tool_name}({args})\n"
                context += f"Observation: {observation}\n\n"
            else:
                break
        
        return {"success": False, "error": "Max steps reached", "trace": self.trace}

# 定义工具
def search_tool(query: str) -> str:
    """模拟搜索工具"""
    knowledge_base = {
        "Python": "Python是一种高级编程语言，由Guido van Rossum于1991年创建",
        "AI": "人工智能是计算机科学的一个分支，致力于创建智能机器",
        "ReAct": "ReAct是一种结合推理和行动的智能体框架"
    }
    for key in knowledge_base:
        if key.lower() in query.lower():
            return knowledge_base[key]
    return "未找到相关信息"

def calculate_tool(expression: str) -> float:
    """计算工具"""
    try:
        # 安全的数学表达式求值
        allowed_chars = set('0123456789+-*/(). ')
        if all(c in allowed_chars for c in expression):
            return eval(expression)
        return "Invalid expression"
    # 避免裸 except: 捕获所有异常（包括 KeyboardInterrupt、SystemExit 等），
    # 应明确指定预期的异常类型，提高代码可维护性和调试效率
    except (ValueError, SyntaxError, ZeroDivisionError):
        return "Calculation error"

# 模拟LLM响应
def simulate_llm_response(context: str) -> str:
    """简化的LLM模拟"""
    if "Python" in context and "Observation:" not in context:
        return "Thought: 我需要搜索Python的信息\nAction: search(Python)"
    elif "Python是一种" in context:
        return "Thought: 我已经获得了Python的信息，可以回答了\nAction: finish(Python是由Guido van Rossum于1991年创建的高级编程语言)"
    return "Thought: 任务完成\nAction: finish(完成)"

# 测试ReAct智能体
tools = {
    "search": search_tool,
    "calculate": calculate_tool
}

agent = ReActAgent(tools)
result = agent.run("告诉我关于Python的信息", simulate_llm_response)

print("ReAct执行轨迹：")
for i, step in enumerate(result['trace'], 1):
    if step['type'] == 'thought':
        print(f"\n步骤 {i} - 思考: {step['content']}")
    elif step['type'] == 'action':
        print(f"步骤 {i} - 行动: {step['tool']}({step['args']})")
    elif step['type'] == 'observation':
        print(f"步骤 {i} - 观察: {step['content']}")
    elif step['type'] == 'finish':
        print(f"\n最终答案: {step['answer']}")

print(f"\n总步数: {result.get('steps', 'N/A')}")

In [ ]:
# 🔬 Micro Practice: Tool Registry
# Goal: 工具注册表

class ToolRegistry:
    """工具注册和管理系统"""
    
    def __init__(self):
        self.tools = {}
    
    def register(self, name: str, func: callable, description: str, parameters: Dict):
        """注册工具"""
        self.tools[name] = {
            "function": func,
            "description": description,
            "parameters": parameters
        }
    
    def get_tool_descriptions(self) -> str:
        """获取所有工具的描述（用于提示词）"""
        descriptions = []
        for name, info in self.tools.items():
            params = ', '.join([f"{k}: {v}" for k, v in info['parameters'].items()])
            descriptions.append(f"- {name}({params}): {info['description']}")
        return "\n".join(descriptions)
    
    def execute(self, name: str, **kwargs) -> Any:
        """执行工具"""
        if name not in self.tools:
            raise ValueError(f"Tool '{name}' not found")
        return self.tools[name]["function"](**kwargs)

# 创建工具注册表
registry = ToolRegistry()

# 注册搜索工具
registry.register(
    name="web_search",
    func=lambda query: f"搜索结果: {query} 相关的网页内容...",
    description="在网络上搜索信息",
    parameters={"query": "str"}
)

# 注册计算器
registry.register(
    name="calculator",
    func=lambda expression: eval(expression) if all(c in '0123456789+-*/(). ' for c in expression) else "Invalid",
    description="执行数学计算",
    parameters={"expression": "str"}
)

# 注册代码执行器
registry.register(
    name="code_executor",
    func=lambda code: f"执行代码: {code[:50]}...",
    description="执行Python代码",
    parameters={"code": "str"}
)

# 注册API调用器
registry.register(
    name="api_call",
    func=lambda endpoint, method="GET": f"调用API: {method} {endpoint}",
    description="调用外部API",
    parameters={"endpoint": "str", "method": "str"}
)

print("可用工具列表：")
print(registry.get_tool_descriptions())

print("\n测试工具调用：")
print(registry.execute("calculator", expression="10 + 20 * 3"))
print(registry.execute("web_search", query="人工智能"))

In [ ]:
# 可视化：ReAct执行流程

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：ReAct循环流程
steps = ['Thought', 'Action', 'Observation', 'Thought', 'Action', 'Finish']
y_pos = np.arange(len(steps))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#3498db', '#e74c3c', '#9b59b6']

ax1.barh(y_pos, [1]*len(steps), color=colors, alpha=0.7)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(steps)
ax1.set_xlabel('执行顺序', fontsize=12)
ax1.set_title('ReAct执行流程', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# 添加箭头
for i in range(len(steps)-1):
    ax1.annotate('', xy=(0.5, i+1), xytext=(0.5, i),
                arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# 右图：工具使用统计
tool_names = ['搜索', '计算', '代码执行', 'API调用']
usage_counts = [45, 32, 18, 25]

ax2.bar(tool_names, usage_counts, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], alpha=0.7)
ax2.set_ylabel('使用次数', fontsize=12)
ax2.set_title('工具使用频率', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# 添加数值标签
for i, v in enumerate(usage_counts):
    ax2.text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('react_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ ReAct模式可视化完成")

## 模型上下文协议（MCP）

> 📌 本章基于 MCP 协议 v1.0（2024-11 发布）编写，最新版本请参考 [modelcontextprotocol.io](https://modelcontextprotocol.io)

### MCP概述

模型上下文协议（Model Context Protocol, MCP）是一种标准化的协议，用于大语言模型与外部工具、资源和数据源之间的通信。

### MCP架构

```
┌─────────────┐
│     LLM     │
└──────┬──────┘
       │ MCP Protocol
┌──────▼──────┐
│ MCP Server  │
└──────┬──────┘
       │
   ┌───┴───┬───────┬────────┐
   ▼       ▼       ▼        ▼
Files   Database  API   Resources
```

### MCP核心优势

1. **统一接口**：所有工具使用相同的协议
2. **可组合性**：工具可以灵活组合和扩展
3. **安全性**：内置沙箱和权限控制
4. **跨平台**：支持多种编程语言和环境
5. **标准化**：遵循开放标准，易于集成

### MCP组件

- **Resources**：可访问的数据源（文件、数据库等）
- **Tools**：可执行的函数和操作
- **Prompts**：预定义的提示模板
- **Sampling**：LLM采样配置

### MCP vs 传统API

| 特性 | 传统API | MCP |
|------|---------|-----|
| 接口标准化 | 各异 | 统一 |
| 工具发现 | 手动 | 自动 |
| 安全控制 | 分散 | 集中 |
| 上下文管理 | 有限 | 丰富 |

In [ ]:
# 🔬 Micro Practice: Simplified MCP Server
# Goal: 简化的MCP服务器实现

from typing import List, Dict, Any, Callable
from dataclasses import dataclass, field
import json

@dataclass
class MCPResource:
    """MCP资源定义"""
    uri: str
    name: str
    description: str
    mime_type: str
    
@dataclass
class MCPTool:
    """MCP工具定义"""
    name: str
    description: str
    parameters: Dict[str, Any]
    handler: Callable

class MCPServer:
    """简化的MCP服务器"""
    
    def __init__(self, name: str):
        self.name = name
        self.resources: Dict[str, MCPResource] = {}
        self.tools: Dict[str, MCPTool] = {}
        self.prompts: Dict[str, str] = {}
    
    def register_resource(self, resource: MCPResource):
        """注册资源"""
        self.resources[resource.uri] = resource
    
    def register_tool(self, tool: MCPTool):
        """注册工具"""
        self.tools[tool.name] = tool
    
    def register_prompt(self, name: str, template: str):
        """注册提示模板"""
        self.prompts[name] = template
    
    def list_resources(self) -> List[Dict]:
        """列出所有资源"""
        return [
            {
                "uri": r.uri,
                "name": r.name,
                "description": r.description,
                "mimeType": r.mime_type
            }
            for r in self.resources.values()
        ]
    
    def list_tools(self) -> List[Dict]:
        """列出所有工具"""
        return [
            {
                "name": t.name,
                "description": t.description,
                "parameters": t.parameters
            }
            for t in self.tools.values()
        ]
    
    def read_resource(self, uri: str) -> str:
        """读取资源"""
        if uri not in self.resources:
            raise ValueError(f"Resource not found: {uri}")
        # 简化实现：返回模拟数据
        return f"Content of {self.resources[uri].name}"
    
    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Any:
        """调用工具"""
        if name not in self.tools:
            raise ValueError(f"Tool not found: {name}")
        tool = self.tools[name]
        return tool.handler(**arguments)
    
    def get_prompt(self, name: str, variables: Dict[str, str] = None) -> str:
        """获取提示模板"""
        if name not in self.prompts:
            raise ValueError(f"Prompt not found: {name}")
        template = self.prompts[name]
        if variables:
            for key, value in variables.items():
                template = template.replace(f"{{{key}}}", value)
        return template

# 创建MCP服务器
mcp_server = MCPServer("demo-server")

# 注册文件系统资源
mcp_server.register_resource(MCPResource(
    uri="file:///data/users.json",
    name="用户数据",
    description="用户信息数据库",
    mime_type="application/json"
))

# 注册数据库资源
mcp_server.register_resource(MCPResource(
    uri="db://localhost/products",
    name="产品数据库",
    description="产品信息数据库",
    mime_type="application/sql"
))

# 注册查询工具
def query_database(table: str, filter: str = None) -> List[Dict]:
    """查询数据库"""
    return [{"id": 1, "name": "示例数据", "table": table}]

mcp_server.register_tool(MCPTool(
    name="query_db",
    description="查询数据库",
    parameters={
        "table": {"type": "string", "description": "表名"},
        "filter": {"type": "string", "description": "过滤条件", "optional": True}
    },
    handler=query_database
))

# 注册文件操作工具
def read_file(path: str) -> str:
    """读取文件"""
    return f"文件内容: {path}"

mcp_server.register_tool(MCPTool(
    name="read_file",
    description="读取文件内容",
    parameters={
        "path": {"type": "string", "description": "文件路径"}
    },
    handler=read_file
))

# 注册API调用工具
def call_api(endpoint: str, method: str = "GET", data: Dict = None) -> Dict:
    """调用外部API"""
    return {"status": "success", "endpoint": endpoint, "method": method}

mcp_server.register_tool(MCPTool(
    name="api_call",
    description="调用外部API",
    parameters={
        "endpoint": {"type": "string", "description": "API端点"},
        "method": {"type": "string", "description": "HTTP方法", "optional": True},
        "data": {"type": "object", "description": "请求数据", "optional": True}
    },
    handler=call_api
))

# 注册提示模板
mcp_server.register_prompt(
    "data_analysis",
    "分析以下数据并提供见解：\n数据源: {source}\n分析目标: {goal}"
)

print("MCP服务器初始化完成")
print(f"\n可用资源数: {len(mcp_server.resources)}")
print(f"可用工具数: {len(mcp_server.tools)}")
print(f"可用提示数: {len(mcp_server.prompts)}")

In [ ]:
# 🔬 Micro Practice: Agent-MCP Integration
# Goal: 智能体与MCP集成

class MCPAgent:
    """集成MCP的智能体"""
    
    def __init__(self, mcp_server: MCPServer):
        self.mcp = mcp_server
        self.context = []
    
    def discover_tools(self) -> List[Dict]:
        """发现可用工具"""
        tools = self.mcp.list_tools()
        print("发现的工具：")
        for tool in tools:
            print(f"  - {tool['name']}: {tool['description']}")
        return tools
    
    def discover_resources(self) -> List[Dict]:
        """发现可用资源"""
        resources = self.mcp.list_resources()
        print("发现的资源：")
        for resource in resources:
            print(f"  - {resource['name']} ({resource['uri']})")
        return resources
    
    def use_tool(self, tool_name: str, **kwargs) -> Any:
        """使用工具"""
        print(f"\n调用工具: {tool_name}")
        print(f"参数: {kwargs}")
        
        result = self.mcp.call_tool(tool_name, kwargs)
        
        self.context.append({
            "type": "tool_call",
            "tool": tool_name,
            "arguments": kwargs,
            "result": result
        })
        
        print(f"结果: {result}")
        return result
    
    def access_resource(self, uri: str) -> str:
        """访问资源"""
        print(f"\n访问资源: {uri}")
        content = self.mcp.read_resource(uri)
        
        self.context.append({
            "type": "resource_access",
            "uri": uri,
            "content": content
        })
        
        return content
    
    def execute_task(self, task: str) -> Dict[str, Any]:
        """执行任务（使用MCP工具）"""
        print(f"任务: {task}\n")
        
        # 发现可用工具和资源
        self.discover_tools()
        print()
        self.discover_resources()
        
        # 模拟任务执行
        if "查询" in task or "数据库" in task:
            result = self.use_tool("query_db", table="users", filter="active=true")
        elif "文件" in task:
            result = self.use_tool("read_file", path="/data/config.json")
        elif "API" in task:
            result = self.use_tool("api_call", endpoint="https://api.example.com/data")
        else:
            result = "任务类型未识别"
        
        return {
            "task": task,
            "result": result,
            "context_size": len(self.context)
        }

# 测试MCP智能体
agent = MCPAgent(mcp_server)
result = agent.execute_task("查询数据库中的用户信息")

print(f"\n任务完成")
print(f"上下文记录数: {result['context_size']}")

In [ ]:
# 可视化：MCP架构和工具使用

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：MCP组件分布
components = ['Resources', 'Tools', 'Prompts', 'Sampling']
counts = [2, 3, 1, 1]
colors_comp = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

wedges, texts, autotexts = ax1.pie(counts, labels=components, colors=colors_comp,
                                     autopct='%1.0f%%', startangle=90)
ax1.set_title('MCP组件分布', fontsize=14, fontweight='bold')

# 右图：工具调用延迟对比
scenarios = ['直接调用', 'MCP标准化', 'MCP+缓存']
latencies = [150, 180, 45]  # 毫秒
colors_lat = ['#e74c3c', '#f39c12', '#2ecc71']

bars = ax2.bar(scenarios, latencies, color=colors_lat, alpha=0.7)
ax2.set_ylabel('延迟 (ms)', fontsize=12)
ax2.set_title('工具调用延迟对比', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}ms', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('mcp_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ MCP架构可视化完成")

## 智能体规划与记忆

### 规划策略

智能体需要将复杂任务分解为可执行的步骤。主要规划策略包括：

#### 1. Chain-of-Thought (CoT) - 思维链

顺序推理，逐步解决问题：
```
问题 → 步骤1 → 步骤2 → 步骤3 → 答案
```

#### 2. Tree-of-Thoughts (ToT) - 思维树

探索多条路径，选择最优解：
```
        问题
       /  |  \
     方案A B  C
     / \  |  / \
   A1 A2 B1 C1 C2
```

#### 3. Plan-and-Execute - 计划执行

先制定高层计划，再逐步执行：
```
1. 分析任务 → 生成计划
2. 执行步骤1 → 验证结果
3. 执行步骤2 → 验证结果
4. 如需调整 → 重新规划
```

### 记忆系统

智能体需要维护不同类型的记忆：

#### 1. 短期记忆（Short-term Memory）
- 当前对话上下文
- 最近的观察和行动
- 临时工作状态

#### 2. 长期记忆（Long-term Memory）
- 历史经验和知识
- 成功/失败案例
- 向量数据库存储

#### 3. 工作记忆（Working Memory）
- 当前任务状态
- 中间计算结果
- 待执行的计划

### 记忆架构

```
┌─────────────────────────────────┐
│      短期记忆 (对话上下文)        │
└────────────┬────────────────────┘
             │
┌────────────▼────────────────────┐
│      工作记忆 (任务状态)          │
└────────────┬────────────────────┘
             │
┌────────────▼────────────────────┐
│   长期记忆 (向量数据库/知识库)    │
└─────────────────────────────────┘
```

In [ ]:
# 🔬 Micro Practice: Planning Agent Implementation
# Goal: 规划智能体实现

from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum

class PlanStatus(Enum):
    """计划状态"""
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"

@dataclass
class PlanStep:
    """计划步骤"""
    id: int
    description: str
    status: PlanStatus = PlanStatus.PENDING
    result: Optional[Any] = None
    dependencies: List[int] = None
    
    def __post_init__(self):
        if self.dependencies is None:
            self.dependencies = []

class PlanningAgent:
    """具有规划能力的智能体"""
    
    def __init__(self, name: str):
        self.name = name
        self.plan: List[PlanStep] = []
        self.execution_log = []
    
    def decompose_task(self, task: str) -> List[PlanStep]:
        """任务分解"""
        print(f"[{self.name}] 分解任务: {task}")
        
        # 简化的任务分解逻辑
        if "研究" in task or "调研" in task:
            steps = [
                PlanStep(1, "确定研究主题和范围"),
                PlanStep(2, "搜索相关资料", dependencies=[1]),
                PlanStep(3, "分析和总结信息", dependencies=[2]),
                PlanStep(4, "生成研究报告", dependencies=[3])
            ]
        elif "开发" in task or "编程" in task:
            steps = [
                PlanStep(1, "理解需求"),
                PlanStep(2, "设计架构", dependencies=[1]),
                PlanStep(3, "编写代码", dependencies=[2]),
                PlanStep(4, "测试验证", dependencies=[3]),
                PlanStep(5, "部署上线", dependencies=[4])
            ]
        elif "分析" in task:
            steps = [
                PlanStep(1, "收集数据"),
                PlanStep(2, "数据清洗", dependencies=[1]),
                PlanStep(3, "探索性分析", dependencies=[2]),
                PlanStep(4, "生成可视化", dependencies=[3]),
                PlanStep(5, "撰写分析报告", dependencies=[4])
            ]
        else:
            steps = [
                PlanStep(1, "分析任务"),
                PlanStep(2, "执行操作", dependencies=[1]),
                PlanStep(3, "验证结果", dependencies=[2])
            ]
        
        self.plan = steps
        print(f"生成计划，共 {len(steps)} 个步骤")
        return steps
    
    def can_execute(self, step: PlanStep) -> bool:
        """检查步骤是否可以执行（依赖是否满足）"""
        for dep_id in step.dependencies:
            dep_step = next((s for s in self.plan if s.id == dep_id), None)
            if dep_step and dep_step.status != PlanStatus.COMPLETED:
                return False
        return True
    
    def execute_step(self, step: PlanStep) -> bool:
        """执行单个步骤"""
        if not self.can_execute(step):
            print(f"  步骤 {step.id} 依赖未满足，跳过")
            return False
        
        print(f"  执行步骤 {step.id}: {step.description}")
        step.status = PlanStatus.IN_PROGRESS
        
        # 模拟执行
        import random
        success = random.random() > 0.1  # 90%成功率
        
        if success:
            step.status = PlanStatus.COMPLETED
            step.result = f"步骤{step.id}完成"
            print(f"    ✓ 完成")
        else:
            step.status = PlanStatus.FAILED
            print(f"    ✗ 失败")
        
        self.execution_log.append({
            "step_id": step.id,
            "description": step.description,
            "status": step.status.value,
            "result": step.result
        })
        
        return success
    
    def execute_plan(self) -> Dict[str, Any]:
        """执行整个计划"""
        print(f"\n[{self.name}] 开始执行计划\n")
        
        max_iterations = len(self.plan) * 2  # 防止无限循环
        iteration = 0
        
        while iteration < max_iterations:
            iteration += 1
            
            # 找到可执行的待处理步骤
            pending_steps = [s for s in self.plan if s.status == PlanStatus.PENDING]
            if not pending_steps:
                break
            
            executable_steps = [s for s in pending_steps if self.can_execute(s)]
            if not executable_steps:
                print("  没有可执行的步骤，可能存在循环依赖")
                break
            
            # 执行第一个可执行步骤
            step = executable_steps[0]
            success = self.execute_step(step)
            
            if not success:
                print(f"  步骤 {step.id} 失败，尝试重新规划")
                # 简化处理：标记为完成以继续
                step.status = PlanStatus.COMPLETED
        
        # 统计结果
        completed = sum(1 for s in self.plan if s.status == PlanStatus.COMPLETED)
        failed = sum(1 for s in self.plan if s.status == PlanStatus.FAILED)
        
        print(f"\n计划执行完成: {completed}/{len(self.plan)} 步骤成功")
        
        return {
            "total_steps": len(self.plan),
            "completed": completed,
            "failed": failed,
            "success_rate": completed / len(self.plan) if self.plan else 0
        }

# 测试规划智能体
agent = PlanningAgent("规划助手")
agent.decompose_task("研究人工智能在医疗领域的应用")

print("\n计划详情：")
for step in agent.plan:
    deps = f" (依赖: {step.dependencies})" if step.dependencies else ""
    print(f"  {step.id}. {step.description}{deps}")

result = agent.execute_plan()
print(f"\n成功率: {result['success_rate']:.1%}")

In [ ]:
# 🔬 Micro Practice: Memory System Implementation
# Goal: 记忆系统实现

from collections import deque
from typing import List, Dict, Any, Optional
import numpy as np

class ShortTermMemory:
    """短期记忆（对话上下文）"""
    
    def __init__(self, max_size: int = 10):
        self.max_size = max_size
        self.messages = deque(maxlen=max_size)
    
    def add(self, role: str, content: str):
        """添加消息"""
        self.messages.append({"role": role, "content": content})
    
    def get_context(self) -> List[Dict]:
        """获取上下文"""
        return list(self.messages)
    
    def clear(self):
        """清空记忆"""
        self.messages.clear()

class LongTermMemory:
    """长期记忆（向量存储）"""
    
    def __init__(self, embedding_dim: int = 128):
        self.embedding_dim = embedding_dim
        self.memories = []  # [(embedding, metadata)]
    
    def _embed(self, text: str) -> np.ndarray:
        """简化的文本嵌入（实际应使用真实的嵌入模型）"""
        # 使用文本哈希作为简化嵌入
        np.random.seed(hash(text) % (2**32))
        return np.random.randn(self.embedding_dim)
    
    def store(self, content: str, metadata: Dict = None):
        """存储记忆"""
        embedding = self._embed(content)
        self.memories.append({
            "embedding": embedding,
            "content": content,
            "metadata": metadata or {}
        })
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """检索相关记忆"""
        if not self.memories:
            return []
        
        query_embedding = self._embed(query)
        
        # 计算相似度
        similarities = []
        for memory in self.memories:
            sim = np.dot(query_embedding, memory["embedding"]) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(memory["embedding"])
            )
            similarities.append((sim, memory))
        
        # 返回最相关的记忆
        similarities.sort(reverse=True, key=lambda x: x[0])
        return [{
            "content": mem["content"],
            "metadata": mem["metadata"],
            "similarity": sim
        } for sim, mem in similarities[:top_k]]

class WorkingMemory:
    """工作记忆（任务状态）"""
    
    def __init__(self):
        self.current_task = None
        self.task_state = {}
        self.intermediate_results = {}
    
    def set_task(self, task: str):
        """设置当前任务"""
        self.current_task = task
        self.task_state = {"status": "active", "progress": 0}
    
    def update_state(self, key: str, value: Any):
        """更新任务状态"""
        self.task_state[key] = value
    
    def store_result(self, key: str, result: Any):
        """存储中间结果"""
        self.intermediate_results[key] = result
    
    def get_result(self, key: str) -> Optional[Any]:
        """获取中间结果"""
        return self.intermediate_results.get(key)
    
    def clear(self):
        """清空工作记忆"""
        self.current_task = None
        self.task_state = {}
        self.intermediate_results = {}

class MemoryAgent:
    """具有完整记忆系统的智能体"""
    
    def __init__(self, name: str):
        self.name = name
        self.short_term = ShortTermMemory(max_size=10)
        self.long_term = LongTermMemory(embedding_dim=128)
        self.working = WorkingMemory()
    
    def process_message(self, user_message: str) -> str:
        """处理用户消息"""
        # 添加到短期记忆
        self.short_term.add("user", user_message)
        
        # 从长期记忆检索相关信息
        relevant_memories = self.long_term.retrieve(user_message, top_k=2)
        
        # 生成响应（简化）
        response = f"处理消息: {user_message}"
        if relevant_memories:
            response += f" (找到 {len(relevant_memories)} 条相关记忆)"
        
        # 添加响应到短期记忆
        self.short_term.add("assistant", response)
        
        # 存储到长期记忆
        self.long_term.store(
            f"Q: {user_message} A: {response}",
            metadata={"type": "conversation"}
        )
        
        return response
    
    def execute_task(self, task: str) -> Dict[str, Any]:
        """执行任务（使用工作记忆）"""
        # 设置工作记忆
        self.working.set_task(task)
        
        # 模拟任务执行
        steps = ["分析", "执行", "验证"]
        for i, step in enumerate(steps, 1):
            self.working.update_state("progress", i / len(steps))
            self.working.store_result(f"step_{i}", f"{step}完成")
        
        # 完成任务
        self.working.update_state("status", "completed")
        
        # 存储经验到长期记忆
        self.long_term.store(
            f"任务: {task} - 成功完成",
            metadata={"type": "experience", "success": True}
        )
        
        return {
            "task": task,
            "status": self.working.task_state["status"],
            "progress": self.working.task_state["progress"]
        }

# 测试记忆系统
agent = MemoryAgent("记忆助手")

print("对话测试：")
messages = [
    "你好，我想了解机器学习",
    "什么是深度学习？",
    "能再讲讲机器学习吗？"  # 应该检索到第一条
]

for msg in messages:
    response = agent.process_message(msg)
    print(f"用户: {msg}")
    print(f"助手: {response}\n")

print(f"短期记忆条目数: {len(agent.short_term.get_context())}")
print(f"长期记忆条目数: {len(agent.long_term.memories)}")

print("\n任务执行测试：")
result = agent.execute_task("分析用户行为数据")
print(f"任务: {result['task']}")
print(f"状态: {result['status']}")
print(f"进度: {result['progress']:.0%}")

In [ ]:
# 可视化：规划策略和记忆系统

fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# 左图：规划策略对比
ax1 = fig.add_subplot(gs[0, 0])
strategies = ['CoT', 'ToT', 'Plan-Execute']
success_rates = [0.75, 0.85, 0.92]
colors_strat = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax1.barh(strategies, success_rates, color=colors_strat, alpha=0.7)
ax1.set_xlabel('成功率', fontsize=11)
ax1.set_title('规划策略效果对比', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.grid(axis='x', alpha=0.3)

for i, (bar, rate) in enumerate(zip(bars, success_rates)):
    ax1.text(rate + 0.02, i, f'{rate:.0%}', va='center', fontweight='bold')

# 中图：记忆类型容量
ax2 = fig.add_subplot(gs[0, 1])
memory_types = ['短期', '工作', '长期']
capacities = [10, 50, 10000]
colors_mem = ['#f39c12', '#9b59b6', '#1abc9c']

ax2.bar(memory_types, np.log10(capacities), color=colors_mem, alpha=0.7)
ax2.set_ylabel('容量 (log10)', fontsize=11)
ax2.set_title('记忆系统容量', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for i, (cap, color) in enumerate(zip(capacities, colors_mem)):
    ax2.text(i, np.log10(cap) + 0.1, str(cap), ha='center', fontweight='bold')

# 右图：任务执行时间线
ax3 = fig.add_subplot(gs[0, 2])
steps = ['分析', '规划', '执行1', '执行2', '执行3', '验证']
times = [2, 3, 5, 4, 6, 2]  # 秒
cumulative = np.cumsum([0] + times)

colors_time = ['#3498db', '#e74c3c', '#2ecc71', '#2ecc71', '#2ecc71', '#f39c12']
for i, (step, start, duration, color) in enumerate(zip(steps, cumulative[:-1], times, colors_time)):
    ax3.barh(i, duration, left=start, color=color, alpha=0.7)
    ax3.text(start + duration/2, i, step, ha='center', va='center', fontweight='bold', fontsize=9)

ax3.set_yticks([])
ax3.set_xlabel('时间 (秒)', fontsize=11)
ax3.set_title('任务执行时间线', fontsize=13, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('planning_memory.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ 规划与记忆可视化完成")

## 多智能体系统

### 多智能体协作模式

多个智能体可以通过不同的模式协作完成复杂任务：

#### 1. 层级模式（Hierarchical）

管理者智能体将任务分配给工作者智能体：
```
      Manager
     /   |   \
  Agent1 Agent2 Agent3
```

**优势**：清晰的职责划分，易于管理
**适用**：复杂任务分解，需要协调的场景

#### 2. 协作模式（Collaborative）

智能体平等协作，共同完成任务：
```
Agent1 ←→ Agent2 ←→ Agent3
   ↓         ↓         ↓
      Shared Memory
```

**优势**：灵活性高，充分利用各智能体专长
**适用**：需要多角度分析的任务

#### 3. 竞争模式（Competitive）

多个智能体独立解决问题，通过投票或评分选择最佳方案：
```
Agent1 → Solution1 ↘
Agent2 → Solution2 → Evaluator → Best Solution
Agent3 → Solution3 ↗
```

**优势**：提高方案质量，减少单点失败
**适用**：需要高质量输出的场景

#### 4. 流水线模式（Sequential）

智能体按顺序处理任务，每个专注于特定阶段：
```
Input → Agent1 → Agent2 → Agent3 → Output
       (收集)   (分析)   (生成)
```

**优势**：专业化分工，高效处理
**适用**：多阶段处理流程

### 智能体通信

多智能体系统需要有效的通信机制：

1. **消息传递**：直接发送消息
2. **共享内存**：通过共享数据结构交换信息
3. **事件驱动**：基于事件触发协作
4. **协议约定**：遵循统一的通信协议

### 协调机制

- **任务分配**：如何分配子任务
- **冲突解决**：处理智能体间的冲突
- **资源管理**：共享资源的访问控制
- **同步机制**：协调智能体的执行顺序

In [ ]:
# 🔬 Micro Practice: Multi-Agent System Implementation
# Goal: 多智能体系统实现

from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum
import time

class AgentRole(Enum):
    """智能体角色"""
    MANAGER = "manager"
    RESEARCHER = "researcher"
    CODER = "coder"
    REVIEWER = "reviewer"
    COORDINATOR = "coordinator"

@dataclass
class Message:
    """智能体间的消息"""
    sender: str
    receiver: str
    content: str
    message_type: str = "info"
    timestamp: float = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = time.time()

class SharedMemory:
    """智能体共享内存"""
    
    def __init__(self):
        self.data = {}
        self.messages = []
    
    def write(self, key: str, value: Any):
        """写入数据"""
        self.data[key] = value
    
    def read(self, key: str) -> Optional[Any]:
        """读取数据"""
        return self.data.get(key)
    
    def post_message(self, message: Message):
        """发布消息"""
        self.messages.append(message)
    
    def get_messages(self, receiver: str = None) -> List[Message]:
        """获取消息"""
        if receiver:
            return [m for m in self.messages if m.receiver == receiver or m.receiver == "all"]
        return self.messages

class BaseAgent:
    """基础智能体类"""
    
    def __init__(self, name: str, role: AgentRole, shared_memory: SharedMemory):
        self.name = name
        self.role = role
        self.shared_memory = shared_memory
        self.local_memory = []
    
    def send_message(self, receiver: str, content: str, msg_type: str = "info"):
        """发送消息"""
        message = Message(self.name, receiver, content, msg_type)
        self.shared_memory.post_message(message)
        print(f"[{self.name}] → [{receiver}]: {content}")
    
    def receive_messages(self) -> List[Message]:
        """接收消息"""
        return self.shared_memory.get_messages(self.name)
    
    def process(self, task: str) -> str:
        """处理任务（子类实现）"""
        raise NotImplementedError

class ResearcherAgent(BaseAgent):
    """研究员智能体"""
    
    def process(self, task: str) -> str:
        """研究任务"""
        print(f"[{self.name}] 开始研究: {task}")
        # 模拟研究过程
        result = f"研究报告: {task} 的相关资料和分析"
        self.shared_memory.write(f"research_{self.name}", result)
        return result

class CoderAgent(BaseAgent):
    """编程智能体"""
    
    def process(self, task: str) -> str:
        """编写代码"""
        print(f"[{self.name}] 开始编码: {task}")
        # 模拟编码过程
        code = f"def solution():\n    # 实现 {task}\n    pass"
        self.shared_memory.write(f"code_{self.name}", code)
        return code

class ReviewerAgent(BaseAgent):
    """审查员智能体"""
    
    def process(self, task: str) -> str:
        """审查工作"""
        print(f"[{self.name}] 开始审查: {task}")
        # 检查共享内存中的代码
        code = self.shared_memory.read("code_Coder-1")
        if code:
            review = f"审查通过: 代码质量良好"
        else:
            review = f"审查失败: 未找到代码"
        return review

class ManagerAgent(BaseAgent):
    """管理者智能体"""
    
    def __init__(self, name: str, shared_memory: SharedMemory, workers: List[BaseAgent]):
        super().__init__(name, AgentRole.MANAGER, shared_memory)
        self.workers = workers
    
    def delegate_task(self, task: str) -> Dict[str, Any]:
        """分配任务"""
        print(f"\n[{self.name}] 收到任务: {task}")
        print(f"[{self.name}] 分解任务并分配给 {len(self.workers)} 个工作者\n")
        
        results = {}
        
        # 分配给不同的工作者
        for worker in self.workers:
            if worker.role == AgentRole.RESEARCHER:
                subtask = f"研究{task}的背景和方法"
            elif worker.role == AgentRole.CODER:
                subtask = f"实现{task}的代码"
            elif worker.role == AgentRole.REVIEWER:
                subtask = f"审查{task}的实现"
            else:
                subtask = task
            
            self.send_message(worker.name, subtask, "task")
            result = worker.process(subtask)
            results[worker.name] = result
        
        print(f"\n[{self.name}] 所有子任务完成")
        return results
    
    def process(self, task: str) -> str:
        """处理任务"""
        results = self.delegate_task(task)
        summary = f"任务'{task}'完成，{len(results)}个子任务成功执行"
        return summary

# 创建多智能体系统
shared_mem = SharedMemory()

# 创建工作者智能体
researcher = ResearcherAgent("Researcher-1", AgentRole.RESEARCHER, shared_mem)
coder = CoderAgent("Coder-1", AgentRole.CODER, shared_mem)
reviewer = ReviewerAgent("Reviewer-1", AgentRole.REVIEWER, shared_mem)

# 创建管理者智能体
manager = ManagerAgent("Manager", shared_mem, [researcher, coder, reviewer])

# 执行任务
result = manager.process("构建推荐系统")

print(f"\n最终结果: {result}")
print(f"共享内存数据项: {len(shared_mem.data)}")
print(f"消息总数: {len(shared_mem.messages)}")

In [ ]:
# 🔬 Micro Practice: Collaborative Multi-Agent
# Goal: 协作式多智能体

class CollaborativeSystem:
    """协作式多智能体系统"""
    
    def __init__(self):
        self.shared_memory = SharedMemory()
        self.agents = []
    
    def add_agent(self, agent: BaseAgent):
        """添加智能体"""
        self.agents.append(agent)
    
    def broadcast(self, sender: str, content: str):
        """广播消息"""
        message = Message(sender, "all", content, "broadcast")
        self.shared_memory.post_message(message)
        print(f"[{sender}] 广播: {content}")
    
    def collaborate(self, task: str) -> List[str]:
        """协作完成任务"""
        print(f"协作任务: {task}\n")
        
        results = []
        
        # 第一轮：每个智能体独立处理
        print("=== 第一轮：独立处理 ===")
        for agent in self.agents:
            result = agent.process(task)
            results.append(result)
            self.broadcast(agent.name, f"我完成了: {result[:50]}...")
        
        # 第二轮：智能体间交流和改进
        print("\n=== 第二轮：交流改进 ===")
        for agent in self.agents:
            # 查看其他智能体的结果
            other_results = [r for i, r in enumerate(results) if self.agents[i] != agent]
            if other_results:
                agent.send_message("all", f"我看到了其他智能体的工作，很有启发")
        
        return results

# 创建协作系统
collab_system = CollaborativeSystem()

# 添加多个智能体
collab_system.add_agent(ResearcherAgent("Researcher-A", AgentRole.RESEARCHER, collab_system.shared_memory))
collab_system.add_agent(ResearcherAgent("Researcher-B", AgentRole.RESEARCHER, collab_system.shared_memory))
collab_system.add_agent(CoderAgent("Coder-A", AgentRole.CODER, collab_system.shared_memory))

# 协作完成任务
results = collab_system.collaborate("设计聊天机器人架构")

print(f"\n协作完成，共产生 {len(results)} 个输出")

In [ ]:
# 可视化：多智能体系统

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：不同协作模式的效率对比
patterns = ['单智能体', '层级', '协作', '竞争', '流水线']
efficiency = [60, 75, 85, 80, 90]
complexity = [20, 50, 70, 65, 55]

x = np.arange(len(patterns))
width = 0.35

bars1 = ax1.bar(x - width/2, efficiency, width, label='效率', color='#2ecc71', alpha=0.7)
bars2 = ax1.bar(x + width/2, complexity, width, label='复杂度', color='#e74c3c', alpha=0.7)

ax1.set_ylabel('分数', fontsize=11)
ax1.set_title('多智能体协作模式对比', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(patterns, rotation=15, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 右图：智能体通信量
agents = ['Manager', 'Researcher', 'Coder', 'Reviewer']
sent = [12, 5, 6, 4]
received = [8, 10, 9, 11]

x2 = np.arange(len(agents))
ax2.barh(x2 - width/2, sent, width, label='发送', color='#3498db', alpha=0.7)
ax2.barh(x2 + width/2, received, width, label='接收', color='#f39c12', alpha=0.7)

ax2.set_xlabel('消息数量', fontsize=11)
ax2.set_title('智能体通信统计', fontsize=13, fontweight='bold')
ax2.set_yticks(x2)
ax2.set_yticklabels(agents)
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('multiagent_system.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ 多智能体系统可视化完成")

## 智能体评估与安全

### 评估指标

评估智能体性能需要多维度的指标：

#### 1. 任务成功率（Task Success Rate）
- 完成任务的比例
- 是否达到预期目标

#### 2. 效率（Efficiency）
- 完成任务所需步骤数
- 执行时间
- 资源消耗（API调用次数、成本）

#### 3. 工具使用准确性（Tool Usage Accuracy）
- 选择正确工具的比例
- 工具参数正确性
- 工具调用成功率

#### 4. 推理质量（Reasoning Quality）
- 思考过程的逻辑性
- 决策的合理性
- 错误恢复能力

### 评估方法

1. **基准测试**：标准任务集评估
2. **人工评估**：专家打分
3. **A/B测试**：对比不同版本
4. **长期监控**：生产环境表现

In [ ]:
# 🔬 Micro Practice: Agent Evaluation Framework
# Goal: 智能体评估框架

from typing import List, Dict, Any, Callable
from dataclasses import dataclass
import time

@dataclass
class EvaluationResult:
    """评估结果"""
    task_id: str
    success: bool
    steps: int
    time_taken: float
    tool_calls: int
    reasoning_score: float
    error_message: str = None

class AgentEvaluator:
    """智能体评估器"""
    
    def __init__(self):
        self.results: List[EvaluationResult] = []
    
    def evaluate_task(self, agent: Any, task: str, expected_output: Any = None) -> EvaluationResult:
        """评估单个任务"""
        start_time = time.time()
        
        try:
            # 执行任务
            result = agent.execute(task) if hasattr(agent, 'execute') else agent.process(task)
            
            # 计算指标
            steps = getattr(agent, 'step_count', 0)
            tool_calls = len([m for m in getattr(agent, 'memory', []) if m.get('type') == 'action'])
            
            # 简化的推理质量评分
            reasoning_score = 0.8 if result else 0.3
            
            # 判断成功
            success = result is not None
            if expected_output:
                success = success and (str(expected_output) in str(result))
            
            eval_result = EvaluationResult(
                task_id=task[:30],
                success=success,
                steps=steps,
                time_taken=time.time() - start_time,
                tool_calls=tool_calls,
                reasoning_score=reasoning_score
            )
        except Exception as e:
            eval_result = EvaluationResult(
                task_id=task[:30],
                success=False,
                steps=0,
                time_taken=time.time() - start_time,
                tool_calls=0,
                reasoning_score=0.0,
                error_message=str(e)
            )
        
        self.results.append(eval_result)
        return eval_result

In [ ]:
    def get_statistics(self) -> Dict[str, Any]:
        """获取评估统计"""
        if not self.results:
            return {}
        
        total = len(self.results)
        successful = sum(1 for r in self.results if r.success)
        
        return {
            "total_tasks": total,
            "success_rate": successful / total,
            "avg_steps": sum(r.steps for r in self.results) / total,
            "avg_time": sum(r.time_taken for r in self.results) / total,
            "avg_tool_calls": sum(r.tool_calls for r in self.results) / total,
            "avg_reasoning_score": sum(r.reasoning_score for r in self.results) / total
        }

# 测试评估器
evaluator = AgentEvaluator()

# 创建测试智能体
test_agent = SimpleAgent("测试智能体")

# 评估多个任务
test_tasks = [
    "计算 100 + 200",
    "搜索人工智能信息",
    "分析数据趋势"
]

print("开始评估...\n")
for task in test_tasks:
    result = evaluator.evaluate_task(test_agent, task)
    status = "✓" if result.success else "✗"
    print(f"{status} {result.task_id}: {result.steps}步, {result.time_taken:.3f}秒")

print("\n评估统计：")
stats = evaluator.get_statistics()
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

### 安全机制

智能体系统需要多层安全保障：

#### 1. 工具沙箱（Tool Sandboxing）
- 隔离执行环境
- 限制文件系统访问
- 网络访问控制

#### 2. 行动确认（Action Confirmation）
- 高风险操作需要人工确认
- 提供操作预览
- 支持撤销机制

#### 3. 预算限制（Budget Limits）
- 最大步骤数限制
- API调用次数限制
- 成本上限控制
- 超时保护

#### 4. 有害行动过滤（Harmful Action Filtering）
- 检测危险命令
- 阻止恶意操作
- 内容安全审查

### 安全最佳实践

1. **最小权限原则**：只授予必要权限
2. **审计日志**：记录所有操作
3. **异常检测**：监控异常行为
4. **人机协作**：关键决策需人工参与
5. **定期审查**：持续评估和改进

In [ ]:
# 🔬 Micro Practice: Safety Mechanism Implementation
# Goal: 安全机制实现

from typing import List, Set, Callable
import re

class SafetyGuard:
    """智能体安全防护"""
    
    def __init__(self, max_steps: int = 50, max_cost: float = 10.0):
        self.max_steps = max_steps
        self.max_cost = max_cost
        self.current_steps = 0
        self.current_cost = 0.0
        
        # 危险命令黑名单
        self.dangerous_commands = {
            'rm -rf', 'format', 'delete', 'drop table',
            'shutdown', 'reboot', '__import__'
        }
        
        # 需要确认的高风险操作
        self.high_risk_actions = {
            'delete_file', 'execute_code', 'send_email',
            'make_payment', 'modify_database'
        }
    
    def check_step_limit(self) -> bool:
        """检查步骤限制"""
        self.current_steps += 1
        if self.current_steps > self.max_steps:
            raise RuntimeError(f"超过最大步骤数限制: {self.max_steps}")
        return True
    
    def check_cost_limit(self, cost: float) -> bool:
        """检查成本限制"""
        self.current_cost += cost
        if self.current_cost > self.max_cost:
            raise RuntimeError(f"超过成本限制: ${self.max_cost}")
        return True
    
    def is_dangerous_command(self, command: str) -> bool:
        """检测危险命令"""
        command_lower = command.lower()
        for dangerous in self.dangerous_commands:
            if dangerous in command_lower:
                return True
        return False
    
    def requires_confirmation(self, action: str) -> bool:
        """检查是否需要确认"""
        return action in self.high_risk_actions
    
    def validate_action(self, action: str, params: Dict = None) -> bool:
        """验证行动安全性"""
        # 检查步骤限制
        self.check_step_limit()
        
        # 检查危险命令
        if params and 'command' in params:
            if self.is_dangerous_command(params['command']):
                print(f"⚠️  阻止危险命令: {params['command']}")
                return False
        
        # 检查是否需要确认
        if self.requires_confirmation(action):
            print(f"⚠️  高风险操作 '{action}' 需要人工确认")
            # 实际应用中这里会等待用户确认
            return False
        
        return True
    
    def reset(self):
        """重置计数器"""
        self.current_steps = 0
        self.current_cost = 0.0

class SafeAgent:
    """带安全防护的智能体"""
    
    def __init__(self, name: str, safety_guard: SafetyGuard):
        self.name = name
        self.safety = safety_guard
        self.action_log = []
    
    def execute_action(self, action: str, params: Dict = None) -> str:
        """执行行动（带安全检查）"""
        # 安全验证
        if not self.safety.validate_action(action, params):
            result = f"行动被安全系统阻止: {action}"
            self.action_log.append({"action": action, "blocked": True})
            return result
        
        # 执行行动
        result = f"执行: {action}"
        if params:
            result += f" (参数: {params})"
        
        self.action_log.append({"action": action, "blocked": False, "result": result})
        return result

# 测试安全机制
safety_guard = SafetyGuard(max_steps=5, max_cost=10.0)
safe_agent = SafeAgent("安全助手", safety_guard)

print("测试安全机制:\n")

# 测试1：正常操作
print("1. 正常操作:")
result = safe_agent.execute_action("search", {"query": "AI"})
print(f"   {result}\n")

# 测试2：危险命令
print("2. 危险命令:")
result = safe_agent.execute_action("execute_shell", {"command": "rm -rf /"})
print(f"   {result}\n")

# 测试3：高风险操作
print("3. 高风险操作:")
result = safe_agent.execute_action("delete_file", {"path": "/important.txt"})
print(f"   {result}\n")

# 测试4：步骤限制
print("4. 步骤限制测试:")
try:
    for i in range(10):
        safe_agent.execute_action("step", {"num": i})
except RuntimeError as e:
    print(f"   捕获异常: {e}\n")

print(f"总操作数: {len(safe_agent.action_log)}")
blocked = sum(1 for log in safe_agent.action_log if log.get('blocked'))
print(f"被阻止: {blocked}")

In [ ]:
# 可视化：评估与安全

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：评估指标雷达图
categories = ['成功率', '效率', '工具准确性', '推理质量', '安全性']
values = [0.85, 0.75, 0.90, 0.80, 0.95]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]

ax1 = plt.subplot(121, projection='polar')
ax1.plot(angles, values, 'o-', linewidth=2, color='#3498db')
ax1.fill(angles, values, alpha=0.25, color='#3498db')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories)
ax1.set_ylim(0, 1)
ax1.set_title('智能体评估指标', fontsize=13, fontweight='bold', pad=20)
ax1.grid(True)

# 右图：安全事件统计
ax2 = plt.subplot(122)
events = ['正常操作', '危险命令\n阻止', '高风险\n确认', '超限\n拦截']
counts = [150, 5, 12, 3]
colors_safety = ['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

bars = ax2.bar(events, counts, color=colors_safety, alpha=0.7)
ax2.set_ylabel('事件数量', fontsize=11)
ax2.set_title('安全事件统计', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('evaluation_safety.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ 评估与安全可视化完成")

## 实际应用案例

### 应用场景

AI智能体在多个领域有广泛应用：

1. **研究助手**：文献检索、信息总结、报告生成
2. **编程助手**：代码生成、测试、调试
3. **数据分析**：数据探索、可视化、洞察生成
4. **客户服务**：问题解答、工单处理
5. **内容创作**：文章撰写、图像生成

### 设计原则

- **明确目标**：清晰定义智能体的职责
- **工具选择**：提供必要且安全的工具
- **错误处理**：优雅处理失败情况
- **人机协作**：关键决策保留人工参与
- **持续改进**：基于反馈优化性能

In [ ]:
# 🔬 Micro Practice: Research Assistant Agent
# Goal: 研究助手智能体

class ResearchAssistant:
    """研究助手智能体"""
    
    def __init__(self, name: str):
        self.name = name
        self.knowledge_base = {}
        self.search_history = []
    
    def search_papers(self, query: str, max_results: int = 5) -> List[Dict]:
        """搜索学术论文"""
        print(f"[{self.name}] 搜索论文: {query}")
        
        # 模拟搜索结果
        papers = [
            {
                "title": f"关于{query}的研究 (论文{i+1})",
                "authors": ["作者A", "作者B"],
                "year": 2023 - i,
                "abstract": f"这是关于{query}的研究摘要...",
                "citations": 100 - i*10
            }
            for i in range(min(max_results, 5))
        ]
        
        self.search_history.append({"query": query, "results": len(papers)})
        return papers
    
    def summarize_paper(self, paper: Dict) -> str:
        """总结论文"""
        summary = f"""论文总结：
标题: {paper['title']}
作者: {', '.join(paper['authors'])}
年份: {paper['year']}
引用数: {paper['citations']}
摘要: {paper['abstract']}
"""
        return summary
    
    def generate_report(self, topic: str) -> str:
        """生成研究报告"""
        print(f"\n[{self.name}] 生成研究报告: {topic}\n")
        
        # 步骤1：搜索相关论文
        papers = self.search_papers(topic, max_results=3)
        print(f"找到 {len(papers)} 篇相关论文\n")
        
        # 步骤2：总结每篇论文
        summaries = []
        for i, paper in enumerate(papers, 1):
            print(f"总结论文 {i}/{len(papers)}")
            summary = self.summarize_paper(paper)
            summaries.append(summary)
        
        # 步骤3：生成综合报告
        report = f"""# {topic} 研究报告

## 概述
本报告基于 {len(papers)} 篇学术论文，对{topic}进行综合分析。

## 文献综述
{chr(10).join(summaries)}

## 结论
通过对现有文献的分析，我们发现{topic}是一个活跃的研究领域。

## 参考文献
{chr(10).join([f"{i+1}. {p['title']} ({p['year']})" for i, p in enumerate(papers)])}
"""
        
        print(f"\n报告生成完成，共 {len(report)} 字符\n")
        return report

# 测试研究助手
assistant = ResearchAssistant("研究助手-01")
report = assistant.generate_report("深度学习在医疗诊断中的应用")

print("="*50)
print(report[:500] + "...")
print("="*50)
print(f"\n搜索历史: {len(assistant.search_history)} 次查询")

In [ ]:
# 🔬 Micro Practice: Coding Assistant Agent
# Goal: 编程助手智能体

class CodingAssistant:
    """编程助手智能体"""
    
    def __init__(self, name: str):
        self.name = name
        self.code_history = []
    
    def understand_requirements(self, requirements: str) -> Dict:
        """理解需求"""
        print(f"[{self.name}] 分析需求: {requirements}")
        
        # 简化的需求分析
        analysis = {
            "task": requirements,
            "complexity": "medium",
            "estimated_lines": 50,
            "language": "Python"
        }
        
        print(f"  复杂度: {analysis['complexity']}")
        print(f"  预计代码行数: {analysis['estimated_lines']}")
        return analysis
    
    def write_code(self, requirements: str) -> str:
        """编写代码"""
        print(f"\n[{self.name}] 开始编写代码...")
        
        # 模拟代码生成
        code = f'''def solution():
    """{requirements}"""
    # 实现逻辑
    result = None
    
    # 步骤1: 初始化
    data = []
    
    # 步骤2: 处理
    for item in data:
        # 处理每个项目
        pass
    
    # 步骤3: 返回结果
    return result

if __name__ == "__main__":
    result = solution()
    print(f"结果: {{result}}")'''
        
        self.code_history.append({"requirements": requirements, "code": code})
        print(f"代码生成完成，共 {len(code.split(chr(10)))} 行")
        return code
    
    def run_tests(self, code: str) -> Dict:
        """运行测试"""
        print(f"\n[{self.name}] 运行测试...")
        
        # 模拟测试结果
        test_results = {
            "total": 5,
            "passed": 4,
            "failed": 1,
            "coverage": 0.85
        }
        
        print(f"  测试通过: {test_results['passed']}/{test_results['total']}")
        print(f"  代码覆盖率: {test_results['coverage']:.0%}")
        
        return test_results
    
    def debug(self, error: str) -> str:
        """调试代码"""
        print(f"\n[{self.name}] 调试错误: {error}")
        
        fix = f"修复建议: 检查变量初始化和边界条件"
        print(f"  {fix}")
        return fix
    
    def complete_task(self, requirements: str) -> Dict:
        """完成编程任务"""
        print(f"\n{'='*50}")
        print(f"编程任务: {requirements}")
        print(f"{'='*50}\n")
        
        # 步骤1: 理解需求
        analysis = self.understand_requirements(requirements)
        
        # 步骤2: 编写代码
        code = self.write_code(requirements)
        
        # 步骤3: 运行测试
        test_results = self.run_tests(code)
        
        # 步骤4: 如果有失败，进行调试
        if test_results['failed'] > 0:
            fix = self.debug("测试失败")
        
        return {
            "requirements": requirements,
            "code": code,
            "test_results": test_results,
            "status": "completed"
        }

# 测试编程助手
coder = CodingAssistant("编程助手-01")
result = coder.complete_task("实现一个二分查找算法")

print(f"\n任务状态: {result['status']}")
print(f"代码预览:\n{result['code'][:200]}...")

In [ ]:
# 🔬 Micro Practice: Data Analysis Assistant Agent
# Goal: 数据分析助手智能体

class DataAnalysisAssistant:
    """数据分析助手智能体"""
    
    def __init__(self, name: str):
        self.name = name
        self.data = None
        self.insights = []
    
    def load_data(self, source: str) -> np.ndarray:
        """加载数据"""
        print(f"[{self.name}] 加载数据: {source}")
        
        # 模拟数据加载
        self.data = np.random.randn(100, 5)
        print(f"  数据形状: {self.data.shape}")
        return self.data
    
    def explore_data(self) -> Dict:
        """探索数据"""
        print(f"\n[{self.name}] 探索数据...")
        
        stats = {
            "mean": np.mean(self.data, axis=0),
            "std": np.std(self.data, axis=0),
            "min": np.min(self.data, axis=0),
            "max": np.max(self.data, axis=0)
        }
        
        print(f"  均值范围: [{stats['mean'].min():.2f}, {stats['mean'].max():.2f}]")
        print(f"  标准差范围: [{stats['std'].min():.2f}, {stats['std'].max():.2f}]")
        
        return stats
    
    def visualize(self, chart_type: str = "histogram"):
        """可视化数据"""
        print(f"\n[{self.name}] 生成{chart_type}可视化...")
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # 直方图
        axes[0].hist(self.data[:, 0], bins=20, color='#3498db', alpha=0.7, edgecolor='black')
        axes[0].set_title('特征分布', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('值')
        axes[0].set_ylabel('频数')
        axes[0].grid(alpha=0.3)
        
        # 箱线图
        axes[1].boxplot([self.data[:, i] for i in range(min(5, self.data.shape[1]))],
                       labels=[f'特征{i+1}' for i in range(min(5, self.data.shape[1]))])
        axes[1].set_title('特征箱线图', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('值')
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('data_analysis.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("  可视化完成")
    
    def generate_insights(self) -> List[str]:
        """生成洞察"""
        print(f"\n[{self.name}] 生成数据洞察...")
        
        insights = [
            f"数据集包含 {self.data.shape[0]} 个样本和 {self.data.shape[1]} 个特征",
            f"数据整体呈正态分布",
            f"未发现明显异常值",
            f"特征间相关性较低，适合建模"
        ]
        
        self.insights = insights
        for i, insight in enumerate(insights, 1):
            print(f"  {i}. {insight}")
        
        return insights
    
    def analyze(self, source: str) -> Dict:
        """完整分析流程"""
        print(f"\n{'='*50}")
        print(f"数据分析任务: {source}")
        print(f"{'='*50}\n")
        
        # 步骤1: 加载数据
        self.load_data(source)
        
        # 步骤2: 探索数据
        stats = self.explore_data()
        
        # 步骤3: 可视化
        self.visualize()
        
        # 步骤4: 生成洞察
        insights = self.generate_insights()
        
        return {
            "source": source,
            "stats": stats,
            "insights": insights,
            "status": "completed"
        }

# 测试数据分析助手
analyst = DataAnalysisAssistant("分析助手-01")
result = analyst.analyze("用户行为数据.csv")

print(f"\n分析完成")
print(f"生成洞察数: {len(result['insights'])}")

### 生产环境考虑

将智能体部署到生产环境需要考虑以下方面：

#### 1. 错误处理（Error Handling）
```python
try:
    result = agent.execute(task)
except ToolError as e:
    # 工具调用失败，尝试备用方案
    result = fallback_handler(task)
except TimeoutError:
    # 超时，记录并通知
    logger.error(f"Task timeout: {task}")
```

#### 2. 重试逻辑（Retry Logic）
- 指数退避重试
- 最大重试次数限制
- 幂等性保证

#### 3. 监控（Monitoring）
- 任务成功率
- 平均响应时间
- API调用成本
- 错误率和类型

#### 4. 成本优化（Cost Optimization）
- 缓存常见查询
- 批量处理请求
- 选择合适的模型大小
- 设置预算上限

#### 5. 人机协作（Human-in-the-Loop）
- 关键决策需人工审核
- 提供反馈机制
- 支持人工接管
- 记录人工干预

## 常见问题（FAQ）

### Q1: 何时使用智能体而非RAG？

**使用智能体**：
- 需要多步推理和规划
- 需要调用外部工具和API
- 任务需要动态调整策略
- 需要与环境交互

**使用RAG**：
- 主要是信息检索和问答
- 不需要复杂的推理
- 知识相对静态
- 追求低延迟

### Q2: 如何调试智能体失败？

1. **查看执行轨迹**：检查每步的思考和行动
2. **分析工具调用**：确认工具选择和参数正确
3. **检查提示词**：优化系统提示和示例
4. **简化任务**：将复杂任务分解为子任务
5. **添加日志**：记录详细的执行信息

### Q3: 如何控制智能体行为？

1. **明确的系统提示**：清晰定义角色和职责
2. **工具限制**：只提供必要的工具
3. **输出格式约束**：要求特定的输出格式
4. **示例引导**：提供few-shot示例
5. **安全防护**：实施多层安全机制

### Q4: 智能体的局限性是什么？

1. **推理能力**：受限于底层LLM的能力
2. **可靠性**：可能产生幻觉或错误
3. **成本**：多次LLM调用成本较高
4. **延迟**：多步推理导致响应较慢
5. **可控性**：行为不完全可预测

### Q5: 如何评估智能体性能？

1. **任务基准**：在标准任务集上测试
2. **人工评估**：专家评分和反馈
3. **A/B测试**：对比不同版本
4. **生产指标**：监控实际使用效果
5. **用户满意度**：收集用户反馈

## 总结

### 核心要点

1. **智能体架构**：感知-规划-行动-记忆循环
2. **ReAct模式**：交织推理和行动，提高可解释性
3. **MCP协议**：标准化的工具和资源访问接口
4. **规划策略**：CoT、ToT、Plan-and-Execute
5. **记忆系统**：短期、长期、工作记忆
6. **多智能体**：层级、协作、竞争、流水线模式
7. **评估与安全**：多维度评估，多层安全防护
8. **实际应用**：研究、编程、数据分析等场景

### 最佳实践

1. **明确目标**：清晰定义智能体的职责和边界
2. **工具设计**：提供必要、安全、易用的工具
3. **提示工程**：优化系统提示和few-shot示例
4. **错误处理**：实现健壮的错误恢复机制
5. **人机协作**：关键决策保留人工参与
6. **持续监控**：跟踪性能指标和成本
7. **迭代优化**：基于反馈不断改进

### 下一步学习

- **前端集成**：将智能体集成到Web应用
- **高级规划**：学习更复杂的规划算法
- **多模态智能体**：处理图像、音频等多模态输入
- **强化学习**：通过RL优化智能体策略
- **生产部署**：大规模部署和运维

### 参考资源

- ReAct论文：Synergizing Reasoning and Acting in Language Models
- MCP文档：Model Context Protocol Specification
- LangChain：智能体开发框架
- AutoGPT：自主智能体项目
- AgentBench：智能体评估基准

---

**恭喜！** 你已完成AI智能体系统与模型上下文协议的学习。

智能体代表了AI应用的新范式，从被动响应到主动规划和执行。掌握智能体技术，将帮助你构建更强大、更自主的AI系统。

## 端到端集成：电商客服智能助理

前面的章节分别介绍了 RAG 检索（Module 8.1）和 Agent 推理（本章）。在实际业务中，这些能力需要协同工作。下面通过一个简化的电商客服场景，展示 **RAG 知识检索 + Agent 工具调用 + 响应生成** 的端到端集成方式。

```
用户提问 → 意图识别 → ┬→ 知识检索（RAG）→ 商品/政策信息
                       ├→ 工具调用（Agent）→ 订单查询/退换货
                       └→ 响应生成 → 格式化回复
```

In [ ]:
# 端到端电商客服智能助理 Demo
# 整合 RAG 知识检索 + Agent 工具调用 + 响应生成

# === 模拟知识库（RAG 组件）===
knowledge_base = {
    "退换货政策": "支持7天无理由退换货，商品需保持原包装完好。生鲜食品不支持退货。",
    "运费规则": "订单满99元包邮，不满99元收取8元运费。退货运费由买家承担。",
    "会员权益": "金卡会员享受95折优惠，每月赠送5张优惠券。",
}

def rag_retrieve(query, top_k=1):
    """简化的 RAG 检索：基于关键词匹配知识库"""
    scores = {}
    for title, content in knowledge_base.items():
        # 简单关键词重叠度计算（实际场景用向量检索）
        overlap = len(set(query) & set(title + content))
        scores[title] = overlap
    top_doc = max(scores, key=scores.get)
    return f"[知识库] {top_doc}：{knowledge_base[top_doc]}"

# === 模拟业务工具（Agent 组件）===
order_db = {
    "ORD20240301": {"status": "已发货", "tracking": "SF1234567890", "item": "无线耳机"},
    "ORD20240228": {"status": "已签收", "tracking": "YT9876543210", "item": "手机壳"},
}

def tool_query_order(order_id):
    """查询订单状态"""
    if order_id in order_db:
        info = order_db[order_id]
        return f"订单{order_id}：{info['item']}，状态={info['status']}，快递单号={info['tracking']}"
    return f"未找到订单 {order_id}"

def tool_apply_return(order_id, reason):
    """申请退货"""
    if order_id in order_db and order_db[order_id]["status"] == "已签收":
        return f"退货申请已提交：订单{order_id}，原因={reason}，预计1-3个工作日审核"
    return f"订单{order_id}不满足退货条件（需已签收状态）"

# === 意图识别 + 路由 ===
def classify_intent(query):
    """简化的意图识别（实际场景用 BERT 分类器）"""
    if any(kw in query for kw in ["订单", "快递", "发货", "物流"]):
        return "order_query"
    elif any(kw in query for kw in ["退", "换", "退货", "退款"]):
        return "return_request"
    elif any(kw in query for kw in ["政策", "规则", "会员", "优惠"]):
        return "knowledge_query"
    return "general"

# === 端到端处理流程 ===
def handle_customer_query(query, context=None):
    """完整的客服查询处理流程"""
    intent = classify_intent(query)
    
    if intent == "order_query":
        # 从查询中提取订单号（简化处理）
        order_id = context.get("order_id", "ORD20240301") if context else "ORD20240301"
        tool_result = tool_query_order(order_id)
        response = f"为您查询到以下信息：\n{tool_result}"
    elif intent == "return_request":
        order_id = context.get("order_id", "ORD20240228") if context else "ORD20240228"
        reason = context.get("reason", "不满意") if context else "不满意"
        tool_result = tool_apply_return(order_id, reason)
        response = f"{tool_result}"
    elif intent == "knowledge_query":
        rag_result = rag_retrieve(query)
        response = f"根据我们的政策：\n{rag_result}"
    else:
        response = "您好，请问有什么可以帮您？"
    
    return {"intent": intent, "response": response}

# === 运行演示 ===
test_conversations = [
    ("我的订单什么时候能到？", {"order_id": "ORD20240301"}),
    ("这个耳机不想要了，能退吗？", {"order_id": "ORD20240228", "reason": "不想要了"}),
    ("你们的退换货政策是什么？", None),
    ("会员有什么优惠？", None),
]

print("=" * 60)
print("电商客服智能助理 - 端到端演示")
print("=" * 60)

for query, ctx in test_conversations:
    result = handle_customer_query(query, ctx)
    print(f"\n用户：{query}")
    print(f"[意图] {result['intent']}")
    print(f"客服：{result['response']}")
    print("-" * 40)

print("\n架构总结：")
print("1. 意图识别：路由到正确的处理分支")
print("2. RAG 检索：政策/FAQ 类问题从知识库获取答案")  
print("3. Agent 工具调用：订单/退货等操作调用业务 API")
print("4. 响应生成：整合信息生成自然语言回复")
print("\n✓ 端到端客服助理演示完成！")

## 思考题参考答案

本章深入学习了 AI 智能体系统与模型上下文协议（MCP）的核心概念与工程实现。以下思考题围绕智能体设计的关键问题展开。

---

### 问题 1：ReAct 模式（Reasoning + Acting）与纯粹的链式思考（Chain-of-Thought）有什么本质区别？在什么场景下 ReAct 更有优势？

**答：**

**本质区别：**

| 维度 | Chain-of-Thought（CoT） | ReAct |
|-----|------------------------|-------|
| **执行方式** | 纯推理，无外部交互 | 推理与行动交替执行 |
| **信息来源** | 仅依赖 LLM 内部知识 | 通过工具调用获取外部信息 |
| **错误修正** | 无法纠错，一次性生成 | 根据观察结果动态调整策略 |
| **适用场景** | 数学推理、逻辑推断 | 需要实时信息或工具执行的任务 |

**ReAct 的执行循环：**

```
Thought: 分析当前状态，决定下一步行动
Action: 调用工具（搜索、计算、API）
Observation: 获取工具执行结果
Thought: 基于观察结果重新推理
...（循环直到任务完成）
```

**ReAct 的优势场景：**

1. **需要实时信息**：天气查询、新闻检索等需要访问外部数据
2. **多步骤任务**：需要依次执行代码、查询数据库、调用 API
3. **长程任务规划**：任务步骤未知，需要根据中间结果动态调整
4. **错误恢复**：工具调用失败时，智能体可以尝试替代方案

**限制**：ReAct 的轨迹长度受上下文窗口限制，且每次工具调用都有延迟，需要权衡推理深度与响应速度。

---

### 问题 2：模型上下文协议（MCP, Model Context Protocol）的设计目标是什么？它如何解决 AI 智能体与外部工具集成的痛点？

**答：**

**MCP 的设计目标：**

MCP 是 Anthropic 提出的开放标准协议，旨在解决 AI 模型与外部工具、数据源集成时的碎片化问题。其核心目标是：

> **统一接口标准**，让 LLM 能够以一致、安全的方式访问任意外部系统。

**传统集成方式的痛点：**

```
传统方式：
LLM A ←→ 自定义工具包 A
LLM B ←→ 自定义工具包 B
（每个 LLM 需要为每个工具单独实现适配层，N×M 的复杂度）

MCP 方式：
LLM ←→ MCP 客户端 ←→ MCP 服务器 ←→ 外部系统
（标准化协议，N+M 的复杂度）
```

**MCP 的核心组件：**

| 组件 | 角色 | 职责 |
|-----|------|------|
| **MCP Server** | 工具/数据提供方 | 暴露工具、资源、提示模板 |
| **MCP Client** | LLM 侧适配层 | 发现工具、调用工具、处理结果 |
| **Transport Layer** | 通信层 | JSON-RPC 2.0 over stdio/HTTP/WebSocket |

**MCP 解决的核心痛点：**

1. **工具发现**：通过 `tools/list` 接口动态获取可用工具列表，无需硬编码
2. **类型安全**：工具的输入/输出有 JSON Schema 定义，减少格式错误
3. **权限隔离**：工具在独立进程中运行，避免安全风险
4. **跨模型复用**：同一 MCP Server 可服务于不同的 LLM，大幅降低集成成本

---

### 问题 3：智能体的记忆系统如何设计？短期记忆、长期记忆和情节记忆分别适用于什么场景？

**答：**

**智能体记忆层次结构：**

```
┌─────────────────────────────────────┐
│     情节记忆（Episodic Memory）       │  ← 具体经历/案例
├─────────────────────────────────────┤
│     语义记忆（Semantic Memory）       │  ← 领域知识/事实
├─────────────────────────────────────┤
│     程序记忆（Procedural Memory）     │  ← 技能/操作步骤
├─────────────────────────────────────┤
│       工作记忆（Working Memory）      │  ← 当前任务上下文
└─────────────────────────────────────┘
```

**各类记忆对比：**

| 记忆类型 | 存储位置 | 持久性 | 适用场景 | 实现方式 |
|---------|---------|--------|---------|---------|
| **短期/工作记忆** | Prompt 上下文 | 会话级别 | 当前对话轮次、任务状态跟踪 | 对话历史列表 |
| **长期/语义记忆** | 向量数据库 | 永久 | 用户偏好、领域知识积累 | RAG + 向量检索 |
| **情节记忆** | 结构化数据库 | 永久 | 成功/失败经验回放，避免重复错误 | 案例存储 + 相似度检索 |
| **程序记忆** | 代码/工作流定义 | 永久 | 固定操作步骤、最佳实践 | 函数库、工具链 |

**设计建议：**

- 短期任务（单次对话）：仅需工作记忆（Prompt 历史）
- 个性化助手：短期 + 长期记忆，用向量数据库存储用户画像
- 自主学习智能体：需要情节记忆，记录过去的成功经验用于未来决策
- 关键窗口管理：上下文超长时，用**记忆压缩（摘要）**替代截断，避免丢失关键信息

---

### 问题 4：多智能体系统（Multi-Agent System）相比单智能体有哪些优势？常见的协作模式有哪些？

**答：**

**多智能体优势：**

1. **分工并行**：不同 Agent 专注于不同子任务，显著降低单个 Agent 的复杂度
2. **能力互补**：研究型 Agent + 执行型 Agent + 评审型 Agent 形成完整流水线
3. **鲁棒性**：多个 Agent 独立验证结果，减少单点错误
4. **可扩展性**：按需增加专业化 Agent，无需修改现有 Agent

**常见协作模式：**

| 模式 | 描述 | 适用场景 |
|-----|------|---------|
| **流水线（Pipeline）** | Agent 顺序执行，前者输出是后者输入 | 数据处理链、文章生成 |
| **辩论（Debate）** | 多个 Agent 对同一问题提出不同观点，汇总最优解 | 复杂决策、代码审查 |
| **监督（Supervisor）** | 主 Agent 分配任务，子 Agent 执行后汇报 | 项目管理、复杂研究 |
| **集成（Ensemble）** | 多 Agent 独立解决问题，投票或评分选择最佳方案 | 高精度要求任务 |
| **竞争（Competition）** | Agent 竞争完成任务，择优录用 | 创意生成、方案设计 |

**挑战与注意事项：**

- **通信开销**：Agent 间传递大量信息会增加延迟和 token 消耗
- **协调复杂性**：需要明确的任务分配协议和冲突解决机制
- **循环依赖**：Agent 间相互等待可能导致死锁，需设计超时机制
- **一致性**：多 Agent 对同一知识库的并发读写需要事务控制

---

### 问题 5：如何评估 AI 智能体的质量？智能体评估与传统 NLP 任务评估有哪些本质不同？

**答：**

**智能体评估的独特挑战：**

传统 NLP 任务（如分类、翻译）有明确的标准答案，但智能体任务往往是**开放式的、过程导向的**，评估需要兼顾最终结果和中间步骤。

**智能体评估维度：**

| 维度 | 评估内容 | 评估方法 |
|-----|---------|---------|
| **任务完成率** | 最终目标是否达成 | 自动验证（代码运行、断言检查）|
| **轨迹质量** | 推理过程是否合理、高效 | 人工评分 / LLM-as-Judge |
| **工具使用效率** | 是否选择了最优工具，调用次数是否最少 | 轨迹分析 |
| **错误恢复能力** | 遇到异常时能否自我纠正 | 注入错误测试 |
| **安全性** | 是否产生有害行为或越权操作 | 红队测试（Red Teaming）|
| **效率** | 完成任务的时间和 token 消耗 | 基准测试对比 |

**主要评估基准：**

- **WebArena / WorkArena**：在真实网络环境中评估任务完成能力
- **AgentBench**：多维度综合评估（OS 操作、数据库查询、网络浏览等）
- **SWE-Bench**：评估代码修复智能体的实际工程能力

**LLM-as-Judge 方法：**

用另一个（更强的）LLM 评估智能体的推理轨迹质量，重点关注：
- 思考过程的逻辑性
- 工具调用的必要性
- 错误处理的优雅程度
- 答案的准确性和完整性

**结论**：智能体评估需要**结果 + 过程 + 安全性**三维综合评估，不能仅看最终输出是否正确。
